## 230968174
## Smeet pancholi
### week6
## 16th feb

In [1]:
# Importing necessary libraries for data handling and model evaluation
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression


Exercise 1: Hill-Climbing Search for Feature Selection in Predictive Modeling

In [2]:
# Load the Iris dataset
data = load_iris()
X = data.data
y = data.target

# Number of features
n_features = X.shape[1]


In [3]:
# Randomly generate an initial state (binary vector)
current_state = np.random.randint(0, 2, size=n_features)


In [4]:
# Function to evaluate model accuracy with selected features using cross-validation
def evaluate_state(state):
    selected_features = [i for i in range(n_features) if state[i] == 1]
    if len(selected_features) == 0:
        return 0  # Return 0 if no features are selected
    X_selected = X[:, selected_features]
    model = LogisticRegression(max_iter=200)
    return np.mean(cross_val_score(model, X_selected, y, cv=5))  # 5-fold cross-validation accuracy


In [5]:
# Steepest Ascent Hill Climbing Algorithm
def hill_climbing():
    current_state = np.random.randint(0, 2, size=n_features)  # Start with a random state
    current_score = evaluate_state(current_state)  # Evaluate the current state
    
    while True:
        neighbors = []
        for i in range(n_features):
            # Flip each bit (add/remove a feature)
            neighbor = current_state.copy()
            neighbor[i] = 1 - neighbor[i]
            neighbors.append((neighbor, evaluate_state(neighbor)))
        
        # Find the best neighbor with the highest score
        best_neighbor, best_score = max(neighbors, key=lambda x: x[1])
        
        if best_score <= current_score:
            break  # Stop if no improvement
        else:
            current_state, current_score = best_neighbor, best_score  # Move to the best neighbor
    
    return current_state, current_score


In [6]:
# Run the hill-climbing algorithm multiple times and store results
for _ in range(5):  # Run it 5 times with different initial states
    best_state, best_accuracy = hill_climbing()
    print(f"Best Feature Subset: {best_state}, Accuracy: {best_accuracy}")


Best Feature Subset: [1 1 1 1], Accuracy: 0.9733333333333334
Best Feature Subset: [1 1 1 1], Accuracy: 0.9733333333333334
Best Feature Subset: [1 1 1 1], Accuracy: 0.9733333333333334
Best Feature Subset: [1 1 1 1], Accuracy: 0.9733333333333334
Best Feature Subset: [1 1 1 1], Accuracy: 0.9733333333333334


Exercise 2: Simulated Annealing for Vehicle Routing Optimization

In [7]:
# Import libraries for simulated annealing and routing problem
import numpy as np
import random
import math


In [8]:
# Define delivery locations (just for simplicity, using 5 random points)
locations = np.array([[0, 0], [1, 2], [2, 4], [3, 1], [4, 3]])

# Initial state: a random permutation of the delivery locations
initial_state = list(range(len(locations)))
random.shuffle(initial_state)


In [9]:
# Cost function: Euclidean distance between two points
def calculate_total_distance(route):
    total_distance = 0
    for i in range(len(route)-1):
        total_distance += np.linalg.norm(locations[route[i]] - locations[route[i+1]])
    total_distance += np.linalg.norm(locations[route[-1]] - locations[route[0]])  # Return to depot
    return total_distance


In [10]:
# Simulated Annealing Implementation
def simulated_annealing(initial_state, T_start=1000, T_end=1, alpha=0.99):
    current_state = initial_state
    current_cost = calculate_total_distance(current_state)
    T = T_start
    
    while T > T_end:
        # Generate neighbor by swapping two locations in the route
        new_state = current_state.copy()
        i, j = random.sample(range(len(locations)), 2)
        new_state[i], new_state[j] = new_state[j], new_state[i]
        
        new_cost = calculate_total_distance(new_state)
        delta_E = new_cost - current_cost
        
        # Acceptance probability
        if delta_E < 0 or random.random() < math.exp(-delta_E / T):
            current_state, current_cost = new_state, new_cost
        
        T *= alpha  # Decrease temperature
        
    return current_state, current_cost


In [11]:
# Run Simulated Annealing and display the results
best_state, best_cost = simulated_annealing(initial_state)
print(f"Best Route: {best_state}, Total Distance: {best_cost}")


Best Route: [1, 4, 2, 0, 3], Total Distance: 15.26882723033592


Exercise 3: Genetic Algorithm for Autonomous Drone Path Planning

In [13]:
# Import libraries for genetic algorithms and path planning
import numpy as np
import random


In [14]:
# Define the 2D map with obstacles
map_size = (10, 10)
obstacles = [(3, 3), (4, 4), (6, 6)]

# Fitness function to calculate total path length and obstacle collisions
def fitness(path):
    total_distance = 0
    for i in range(1, len(path)):
        total_distance += np.linalg.norm(np.array(path[i-1]) - np.array(path[i]))
    collisions = sum([p in obstacles for p in path])  # Count collisions
    return total_distance + collisions * 100  # Penalize collisions heavily


In [15]:
# Genetic Algorithm: Selection, Crossover, and Mutation
def tournament_selection(population, scores, k=3):
    selected = random.sample(list(zip(population, scores)), k)
    selected.sort(key=lambda x: x[1])
    return selected[0][0]  # Return the best individual

def one_point_crossover(parent1, parent2):
    crossover_point = random.randint(1, len(parent1)-2)
    child = parent1[:crossover_point] + parent2[crossover_point:]
    return child

def mutation(path):
    mutation_point = random.randint(1, len(path)-2)
    path[mutation_point] = (random.randint(0, map_size[0]-1), random.randint(0, map_size[1]-1))
    return path


In [16]:
# Genetic Algorithm for Path Planning
def genetic_algorithm(population_size=10, generations=100):
    population = [random.sample(range(map_size[0]), 5) for _ in range(population_size)]
    best_path = None
    best_fitness = float('inf')
    
    for gen in range(generations):
        scores = [fitness(path) for path in population]
        best_idx = np.argmin(scores)
        
        if scores[best_idx] < best_fitness:
            best_fitness = scores[best_idx]
            best_path = population[best_idx]
        
        new_population = []
        for _ in range(population_size//2):
            parent1 = tournament_selection(population, scores)
            parent2 = tournament_selection(population, scores)
            
            child1 = one_point_crossover(parent1, parent2)
            child2 = one_point_crossover(parent2, parent1)
            
            new_population.append(mutation(child1))
            new_population.append(mutation(child2))
        
        population = new_population
        
    return best_path, best_fitness


In [17]:
# Run the genetic algorithm and display the best path
best_path, best_fitness = genetic_algorithm()
print(f"Best Path: {best_path}, Path Length: {best_fitness}")


Best Path: [4, (3, 4), (3, 4), (2, 2), 1], Path Length: 4.650281539872885
